# Bài 4 — Ba cái bẫy của K-Means

Thực hiện theo mục 5: không chuẩn hóa dữ liệu, khởi tạo một lần không phù hợp, và áp dụng K-Means cho cụm không có dạng gần tròn.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
df=pd.read_csv("data/khach_hang.csv")
X=df[["thu_nhap","diem_chi_tieu"]]
Xs=StandardScaler().fit_transform(X)

## Bẫy 1: quên chuẩn hóa
Đổi thu nhập từ triệu sang nghìn đồng để minh họa việc đặc trưng có độ lớn áp đảo khoảng cách.

In [ ]:
X_nghin=X.copy(); X_nghin["thu_nhap"]*=1000
km_tho=KMeans(n_clusters=5,n_init=10,random_state=0).fit(X_nghin)
tam_tho=km_tho.cluster_centers_.copy(); tam_tho[:,0]/=1000
print("Tâm không chuẩn hóa (thu nhập triệu, điểm chi tiêu):")
print(np.round(tam_tho[np.argsort(tam_tho[:,0])],1))

## Bẫy 2: khởi tạo xấu
Chạy nhiều seed với `n_init=1` để thấy nghiệm có thể khác nhau; sau đó dùng nhiều lần khởi tạo và giữ nghiệm tốt nhất.

In [ ]:
inertia=[]
for rs in range(20):
    km=KMeans(n_clusters=5,init="random",n_init=1,random_state=rs).fit(Xs)
    inertia.append(round(km.inertia_,1))
print("Inertia 20 lần:",sorted(inertia))
tot=KMeans(n_clusters=5,n_init=10,random_state=0).fit(Xs)
print("Với n_init=10:",round(tot.inertia_,1))

## Bẫy 3: cụm không tròn
Dữ liệu hai vầng trăng có ranh giới cong. K-Means chia theo khoảng cách tới tâm nên không thể bám sát hình dạng này.

In [ ]:
tr=pd.read_csv("data/hai_vang_trang.csv")
km_tr=KMeans(n_clusters=2,n_init=10,random_state=0)
nhan=km_tr.fit_predict(tr[["x1","x2"]])
khop=(nhan==tr["nhan"]).mean()
print(f"Tỷ lệ khớp sau khi xét hoán đổi nhãn: {max(khop,1-khop):.4f}")
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].scatter(tr.x1,tr.x2,c=tr.nhan,cmap="coolwarm",s=15); ax[0].set_title("Nhãn thật")
ax[1].scatter(tr.x1,tr.x2,c=nhan,cmap="coolwarm",s=15); ax[1].set_title("K-Means")
plt.tight_layout(); plt.show()